In [1]:
# CELL 1 — ENVIRONMENT SETUP

from google.colab import drive
from pathlib import Path
import os
import sys
import subprocess
import time
import json
import shutil

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Data_Lakehouse_Project"
)

APP_DIR = PROJECT_ROOT / "bi" / "streamlit"

APP_FILE = APP_DIR / "app.py"
DATA_ACCESS_FILE = APP_DIR / "data_access.py"
REQUIREMENTS_FILE = APP_DIR / "requirements.txt"

DB_PATH = (
    PROJECT_ROOT / "database" / "credit_lakehouse.duckdb"
)

LOG_PATH = Path("/content/streamlit_dashboard.log")

assert PROJECT_ROOT.is_dir(), (
    f"Không tìm thấy thư mục dự án: {PROJECT_ROOT}"
)

assert APP_FILE.is_file(), (
    f"Không tìm thấy ứng dụng: {APP_FILE}"
)

assert DATA_ACCESS_FILE.is_file(), (
    f"Không tìm thấy module: {DATA_ACCESS_FILE}"
)

assert DB_PATH.is_file(), (
    f"Không tìm thấy database: {DB_PATH}"
)

os.environ["LAKEHOUSE_ROOT"] = str(PROJECT_ROOT)
os.environ["DUCKDB_PATH"] = str(DB_PATH)
os.environ["DB_PATH"] = str(DB_PATH)

print("PROJECT ROOT:", PROJECT_ROOT)
print("APP DIRECTORY:", APP_DIR)
print("DATABASE:", DB_PATH)
print("DATABASE SIZE:", round(DB_PATH.stat().st_size / 1024**3, 3), "GiB")
print("PASS — Đường dẫn tồn tại.")

Mounted at /content/drive
PROJECT ROOT: /content/drive/MyDrive/Data_Lakehouse_Project
APP DIRECTORY: /content/drive/MyDrive/Data_Lakehouse_Project/bi/streamlit
DATABASE: /content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb
DATABASE SIZE: 0.039 GiB
PASS — Đường dẫn tồn tại.


In [2]:
# CELL 2 — INSTALL DEPENDENCIES

import subprocess
import sys

packages = [
    "streamlit",
    "duckdb",
    "pandas",
    "numpy",
    "plotly",
    "pyarrow",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

if REQUIREMENTS_FILE.exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(REQUIREMENTS_FILE),
        ],
        check=True
    )

import streamlit
import duckdb
import pandas
import plotly

print("Streamlit:", streamlit.__version__)
print("DuckDB:", duckdb.__version__)
print("Pandas:", pandas.__version__)

print("PASS — Dependencies installed.")

Streamlit: 1.66.1
DuckDB: 1.5.6
Pandas: 2.2.3
PASS — Dependencies installed.


In [3]:
# CELL 3 — DATABASE VALIDATION

import duckdb
import pandas as pd

con = duckdb.connect(
    str(DB_PATH),
    read_only=True
)

tables = con.execute("""
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema IN (
        'bronze',
        'silver',
        'gold'
    )
    ORDER BY
        table_schema,
        table_name
""").df()

display(tables)

required = [
    ("bronze", "application_train"),
    ("bronze", "application_test"),
    ("bronze", "previous_application"),
    ("bronze", "bureau"),
    ("bronze", "bureau_balance"),
    ("bronze", "installments_payments"),
    ("silver", "stg_application_train"),
    ("silver", "stg_previous_application"),
    ("silver", "stg_bureau"),
    ("gold", "fct_application_train"),
    ("gold", "feature_mart_application_train"),
]

existing = set(
    zip(tables["table_schema"], tables["table_name"])
)

results = []

for schema, table in required:
    found = (schema, table) in existing

    results.append({
        "Schema": schema,
        "Table": table,
        "Status": "FOUND" if found else "MISSING"
    })

validation_df = pd.DataFrame(results)

display(validation_df)

print(
    "Found:",
    (validation_df["Status"] == "FOUND").sum(),
    "/",
    len(validation_df)
)

con.close()

,table_schema,table_name,table_type
0,bronze,application_test,VIEW
1,bronze,application_train,VIEW
2,bronze,bureau,VIEW
3,bronze,bureau_balance,VIEW
4,bronze,credit_card_balance,VIEW
5,bronze,installments_payments,VIEW
6,bronze,pos_cash_balance,VIEW
7,bronze,previous_application,VIEW
8,gold,fct_application_train,BASE TABLE
9,gold,feature_mart_application_train,BASE TABLE


,Schema,Table,Status
0,bronze,application_train,FOUND
1,bronze,application_test,FOUND
2,bronze,previous_application,FOUND
3,bronze,bureau,FOUND
4,bronze,bureau_balance,FOUND
5,bronze,installments_payments,FOUND
6,silver,stg_application_train,FOUND
7,silver,stg_previous_application,FOUND
8,silver,stg_bureau,FOUND
9,gold,fct_application_train,FOUND


Found: 11 / 11


In [4]:
# CELL 4 — SOURCE AND DATA TYPE VALIDATION

import duckdb
import pandas as pd

con = duckdb.connect(
    str(DB_PATH),
    read_only=True
)

sources = [
    "bronze.application_train",
    "bronze.application_test",
    "bronze.previous_application",
    "bronze.bureau",
    "bronze.bureau_balance",
    "bronze.installments_payments",
    "silver.stg_previous_application",
    "silver.stg_bureau",
    "gold.fct_application_train",
    "gold.feature_mart_application_train",
]

checks = []

for source_name in sources:
    try:
        # LIMIT 1 kiểm tra khả năng đọc mà không đếm toàn bộ dữ liệu
        sample = con.execute(
            f"SELECT * FROM {source_name} LIMIT 1"
        ).df()

        checks.append({
            "Source": source_name,
            "Status": "PASS",
            "Columns": len(sample.columns),
            "Error": None
        })

    except Exception as e:
        checks.append({
            "Source": source_name,
            "Status": "FAIL",
            "Columns": None,
            "Error": str(e)[:500]
        })

display(pd.DataFrame(checks))

print("\n===== IMPORTANT DATA TYPES =====")

important_columns = [
    "SK_ID_CURR",
    "SK_ID_PREV",
    "DAYS_DECISION",
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "AMT_PAYMENT",
    "AMT_INSTALMENT",
    "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT",
]

types_df = con.execute("""
    SELECT
        table_schema,
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema IN ('bronze', 'silver', 'gold')
    ORDER BY table_schema, table_name, ordinal_position
""").df()

display(
    types_df[
        types_df["column_name"].isin(important_columns)
    ].reset_index(drop=True)
)

con.close()

,Source,Status,Columns,Error
0,bronze.application_train,PASS,123,None
1,bronze.application_test,PASS,122,None
2,bronze.previous_application,PASS,38,None
3,bronze.bureau,PASS,18,None
4,bronze.bureau_balance,PASS,4,None
5,bronze.installments_payments,PASS,9,None
6,silver.stg_previous_application,PASS,41,None
7,silver.stg_bureau,PASS,22,None
8,gold.fct_application_train,PASS,11,None
9,gold.feature_mart_application_train,PASS,13,None



===== IMPORTANT DATA TYPES =====


,table_schema,table_name,column_name,data_type
0,bronze,application_test,SK_ID_CURR,VARCHAR
1,bronze,application_train,SK_ID_CURR,VARCHAR
2,bronze,bureau,SK_ID_CURR,VARCHAR
3,bronze,bureau,AMT_CREDIT_SUM,VARCHAR
4,bronze,bureau,AMT_CREDIT_SUM_DEBT,VARCHAR
5,bronze,credit_card_balance,SK_ID_PREV,VARCHAR
6,bronze,credit_card_balance,SK_ID_CURR,VARCHAR
7,bronze,installments_payments,SK_ID_PREV,VARCHAR
8,bronze,installments_payments,SK_ID_CURR,VARCHAR
9,bronze,installments_payments,DAYS_INSTALMENT,VARCHAR


In [5]:
# CELL 5 — SAFE APPLICATION PATCH

from pathlib import Path
import ast
import shutil
import difflib
import datetime

original = APP_FILE.read_text(encoding="utf-8")

# Tạo bản sao lưu trong /content, không ghi đè database
backup_dir = Path("/content/streamlit_backups")
backup_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup_file = backup_dir / f"app_{timestamp}.py"

shutil.copy2(APP_FILE, backup_file)

updated = original
changes = []

patches = [
    (
        "DAYS_DECISION between ? and 0",
        "TRY_CAST(DAYS_DECISION AS INTEGER) between ? and 0",
        "Convert DAYS_DECISION"
    ),
    (
        "sum(AMT_CREDIT_SUM_DEBT) debt",
        "sum(TRY_CAST(AMT_CREDIT_SUM_DEBT AS DECIMAL(20,4))) debt",
        "Convert bureau debt"
    ),
]

for old, new, label in patches:
    if old in updated:
        updated = updated.replace(old, new)
        changes.append(label)

# Hiển thị traceback của exception trong giao diện
old_error = (
    "st.error(f'Truy vấn chưa hoàn tất: "
    "{type(e).__name__}: {e}'); st.stop()"
)

new_error = (
    "st.error(f'Truy vấn chưa hoàn tất: "
    "{type(e).__name__}: {e}'); st.exception(e); st.stop()"
)

if old_error in updated:
    updated = updated.replace(old_error, new_error)
    changes.append("Display SQL traceback")

# Kiểm tra cú pháp trước khi ghi vào Drive
ast.parse(updated)

print("===== CHANGES =====")
print(changes if changes else "Không có mẫu sửa nào khớp")

print("\n===== DIFF =====")

diff = difflib.unified_diff(
    original.splitlines(),
    updated.splitlines(),
    fromfile="app.py (original)",
    tofile="app.py (patched)",
    lineterm=""
)

print("\n".join(diff))

if updated != original:
    APP_FILE.write_text(updated, encoding="utf-8")
    print("\nPASS — Đã cập nhật app.py.")
else:
    print("\nINFO — Không có thay đổi.")

print("Backup:", backup_file)

===== CHANGES =====
Không có mẫu sửa nào khớp

===== DIFF =====


INFO — Không có thay đổi.
Backup: /content/streamlit_backups/app_20261105_062249.py


In [6]:
# CELL 6 — SOURCE CODE VALIDATION

import ast
import py_compile

for file in APP_DIR.rglob("*.py"):
    if "__pycache__" in file.parts:
        continue

    py_compile.compile(
        str(file),
        doraise=True
    )
    print("SYNTAX PASS:", file.name)

print("\n===== QUERY FUNCTION INSPECTION =====")

for file in [APP_FILE, DATA_ACCESS_FILE]:

    source = file.read_text(encoding="utf-8")
    lines = source.splitlines()
    tree = ast.parse(source)

    for node in ast.walk(tree):
        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef)
        ) and node.name == "query":

            print("\nFILE:", file.name)

            for i in range(node.lineno, node.end_lineno + 1):
                print(f"{i}: {lines[i-1]}")

print("\n===== IL0C CALLS =====")

app_lines = APP_FILE.read_text(
    encoding="utf-8"
).splitlines()

for number, line in enumerate(app_lines, 1):
    if ".iloc[" in line:
        print(f"{number}: {line.strip()}")

SYNTAX PASS: data_access.py
SYNTAX PASS: app.py

===== QUERY FUNCTION INSPECTION =====

FILE: data_access.py
40: def query(c,sql,params=None):
41:     return c.execute(sql,params or []).fetchdf()

===== IL0C CALLS =====
38: contract=filters[0].multiselect('Loại hợp đồng',query(c,f'select distinct NAME_CONTRACT_TYPE from ({app}) order by 1').iloc[:,0].dropna().tolist())
40: edu=filters[2].multiselect('Học vấn',query(c,f'select distinct NAME_EDUCATION_TYPE from ({app}) order by 1').iloc[:,0].dropna().tolist())
89: contracts=f[1].multiselect('Loại hợp đồng cũ',query(c,'select distinct NAME_CONTRACT_TYPE from previous_application order by 1').iloc[:,0].dropna().tolist() if ready('previous_application') else [])
120: r=query(c,sql,args).iloc[0];metric(cols[3],'04 · Phê duyệt đơn cũ',fmt(r.approved/r.n*100 if r.n else None,'%'),f'{r.approved:,} / {r.n:,}')
157: debt=query(c,'select sum(TRY_CAST(AMT_CREDIT_SUM_DEBT AS DECIMAL(20,4))) debt from bureau where '+restriction,cp).iloc[0,0]


In [7]:
# CELL 7 — SQL SMOKE TEST

import duckdb
import pandas as pd

con = duckdb.connect(
    str(DB_PATH),
    read_only=True
)

sql_tests = {
    "previous_application_status": """
        SELECT NAME_CONTRACT_STATUS
        FROM silver.stg_previous_application
        LIMIT 5
    """,

    "previous_application_days": """
        SELECT
            TRY_CAST(DAYS_DECISION AS INTEGER) AS days_decision
        FROM silver.stg_previous_application
        LIMIT 5
    """,

    "bureau_debt": """
        SELECT
            TRY_CAST(
                AMT_CREDIT_SUM_DEBT AS DECIMAL(20,4)
            ) AS debt
        FROM silver.stg_bureau
        LIMIT 5
    """,

    "previous_application_contract": """
        SELECT DISTINCT NAME_CONTRACT_TYPE
        FROM silver.stg_previous_application
        ORDER BY 1
        LIMIT 10
    """,
}

results = []

for name, sql in sql_tests.items():
    try:
        df = con.execute(sql).df()

        results.append({
            "Test": name,
            "Status": "PASS",
            "Rows": len(df),
            "Error": None
        })

    except Exception as e:
        results.append({
            "Test": name,
            "Status": "FAIL",
            "Rows": None,
            "Error": str(e)[:500]
        })

display(pd.DataFrame(results))

con.close()

,Test,Status,Rows,Error
0,previous_application_status,PASS,5,None
1,previous_application_days,PASS,5,None
2,bureau_debt,PASS,5,None
3,previous_application_contract,PASS,4,None


In [8]:
# CELL 8 — STREAMLIT SERVER

import subprocess
import sys
import os
import time
import signal
from pathlib import Path

PORT = 8501

# Kiểm tra các tiến trình đang dùng cổng
result = subprocess.run(
    ["fuser", "-n", "tcp", str(PORT)],
    capture_output=True,
    text=True
)

pids = [
    int(x)
    for x in result.stdout.split()
    if x.isdigit()
]

for pid in pids:
    cmd = subprocess.run(
        ["ps", "-p", str(pid), "-o", "args="],
        capture_output=True,
        text=True
    ).stdout

    if "streamlit" in cmd.lower():
        os.kill(pid, signal.SIGTERM)
        print("Stopped previous Streamlit:", pid)
    else:
        print("Port used by another process:", pid, cmd)

time.sleep(2)

env = os.environ.copy()
env["LAKEHOUSE_ROOT"] = str(PROJECT_ROOT)
env["DUCKDB_PATH"] = str(DB_PATH)
env["DB_PATH"] = str(DB_PATH)
env["PYTHONPATH"] = str(APP_DIR)

log_handle = open(
    LOG_PATH,
    "w",
    encoding="utf-8"
)

server = subprocess.Popen(
    [
        sys.executable,
        "-m", "streamlit",
        "run",
        str(APP_FILE),

        "--server.address=0.0.0.0",
        f"--server.port={PORT}",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
        "--server.fileWatcherType=none",
        "--browser.gatherUsageStats=false",
    ],
    cwd=str(PROJECT_ROOT),
    env=env,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
)

time.sleep(5)

print("PID:", server.pid)

if server.poll() is None:
    print("PASS — Streamlit process is running.")
else:
    print("FAIL — Streamlit exited.")

print("\n===== SERVER LOG =====")

print(
    LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace"
    )[-4000:]
)

PID: 3110
PASS — Streamlit process is running.

===== SERVER LOG =====



In [9]:
# CELL 9 — HEALTH CHECK AND DASHBOARD URL

import subprocess
from google.colab import output

r = subprocess.run(
    [
        "curl",
        "-sS",
        "--max-time", "10",
        "-o", "/dev/null",
        "-w", "%{http_code}",
        "http://127.0.0.1:8501/_stcore/health",
    ],
    capture_output=True,
    text=True
)

print("HTTP:", r.stdout)
print("Exit code:", r.returncode)
print("Error:", r.stderr or "None")

if r.stdout.strip() == "200":

    url = output.eval_js(
        "google.colab.kernel.proxyPort(8501)"
    )

    print("\nOPEN DASHBOARD:")
    print(url)

else:
    print("Streamlit không phản hồi HTTP 200.")

HTTP: 000
Exit code: 7
Error: curl: (7) Failed to connect to 127.0.0.1 port 8501 after 1 ms: Couldn't connect to server

Streamlit không phản hồi HTTP 200.


In [10]:
# CELL 10 — APPLICATION LOG ANALYSIS

from pathlib import Path

if LOG_PATH.exists():

    logs = LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace"
    )

    keywords = [
        "Traceback",
        "Error",
        "Exception",
        "BinderException",
        "CatalogException",
        "ConversionException",
        "InvalidInputException",
        "Parameter",
        "NoneType",
        "Rejecting WebSocket",
    ]

    selected = [
        line
        for line in logs.splitlines()
        if any(k.lower() in line.lower() for k in keywords)
    ]

    print("Total matched lines:", len(selected))

    for line in selected[-50:]:
        print(line)

else:
    print("Log không tồn tại.")

Total matched lines: 0


In [11]:
# CELL 11 — QA SUMMARY

import json
from pathlib import Path
from datetime import datetime, timezone

qa = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "app_file": str(APP_FILE),
    "database": str(DB_PATH),
    "database_mode": "read_only",
    "checks": {
        "project_exists": PROJECT_ROOT.is_dir(),
        "app_exists": APP_FILE.is_file(),
        "data_access_exists": DATA_ACCESS_FILE.is_file(),
        "database_exists": DB_PATH.is_file(),
        "server_process_alive": server.poll() is None,
        "server_http_200": r.stdout.strip() == "200",
    },
    "not_verified": [
        "WebSocket session established",
        "Dashboard 1 rendered completely",
        "Dashboard 2 rendered completely",
        "KPI independently reconciled",
        "All interactive filters validated",
    ],
}

qa["status"] = (
    "INFRASTRUCTURE_CHECKS_PASSED"
    if all(qa["checks"].values())
    else "NEEDS_REVIEW"
)

qa_path = Path("/content/dashboard_qa.json")

qa_path.write_text(
    json.dumps(qa, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(qa, ensure_ascii=False, indent=2))
print("\nQA file:", qa_path)

{
  "timestamp_utc": "2026-11-05T06:22:59.302432+00:00",
  "project_root": "/content/drive/MyDrive/Data_Lakehouse_Project",
  "app_file": "/content/drive/MyDrive/Data_Lakehouse_Project/bi/streamlit/app.py",
  "database": "/content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb",
  "database_mode": "read_only",
  "checks": {
    "project_exists": true,
    "app_exists": true,
    "data_access_exists": true,
    "database_exists": true,
    "server_process_alive": true,
    "server_http_200": false
  },
  "not_verified": [
    "WebSocket session established",
    "Dashboard 1 rendered completely",
    "Dashboard 2 rendered completely",
    "KPI independently reconciled",
    "All interactive filters validated"
  ],
  "status": "NEEDS_REVIEW"
}

QA file: /content/dashboard_qa.json


In [12]:
# KIỂM THỬ VÀ ĐỐI SOÁT DASHBOARD BI
# Không thay đổi database gốc

from pathlib import Path
from datetime import datetime, timezone
import duckdb
import pandas as pd
import json
import traceback

ROOT = Path("/content/drive/MyDrive/Data_Lakehouse_Project")
DB = ROOT / "database" / "credit_lakehouse.duckdb"
EVIDENCE = ROOT / "evidence" / "dashboard"
EVIDENCE.mkdir(parents=True, exist_ok=True)

assert DB.is_file(), f"Không tìm thấy database: {DB}"

con = duckdb.connect(str(DB), read_only=True)

results = []

def record(test, status, actual=None, expected=None, detail=""):
    results.append({
        "test": test,
        "status": status,
        "actual": actual,
        "expected": expected,
        "detail": detail
    })
    print(f"[{status}] {test}: {actual}")

def execute_test(name, sql, expected=None, tolerance=0):
    try:
        value = con.execute(sql).fetchone()[0]

        if expected is None:
            status = "NOT VERIFIED"
        elif value is not None and abs(float(value) - expected) <= tolerance:
            status = "PASS"
        else:
            status = "FAIL"

        record(name, status, value, expected)
        return value

    except Exception as e:
        record(name, "FAIL", detail=str(e))
        return None


# 1. KIỂM TRA QUY MÔ HỒ SƠ
execute_test(
    "Application Train",
    "SELECT COUNT(*) FROM bronze.application_train",
    307511
)

execute_test(
    "Application Test",
    "SELECT COUNT(*) FROM bronze.application_test",
    48744
)

execute_test(
    "Feature Mart",
    "SELECT COUNT(*) FROM gold.feature_mart_application_train",
    307511
)

# 2. KIỂM TRA TÍNH DUY NHẤT
execute_test(
    "Feature Mart duplicate SK_ID_CURR",
    """
    SELECT COUNT(*) - COUNT(DISTINCT SK_ID_CURR)
    FROM gold.feature_mart_application_train
    """,
    0
)

# 3. TỶ LỆ TARGET
execute_test(
    "TARGET = 1 rate (%)",
    """
    SELECT
        100.0 * COUNT(*) FILTER (WHERE TRY_CAST(TARGET AS INTEGER)=1)
        / NULLIF(COUNT(*) FILTER (
            WHERE TRY_CAST(TARGET AS INTEGER) IN (0,1)
        ), 0)
    FROM gold.fct_application_train
    """,
    8.0729,
    tolerance=0.005
)

# 4. KIỂM TRA CTI TỪ FACT
cti = con.execute("""
    SELECT
        MEDIAN(
            CASE
                WHEN TRY_CAST(AMT_INCOME_TOTAL AS DOUBLE) > 0
                THEN
                    TRY_CAST(AMT_CREDIT AS DOUBLE)
                    / TRY_CAST(AMT_INCOME_TOTAL AS DOUBLE)
            END
        ) AS cti_median
    FROM gold.fct_application_train
""").fetchone()[0]

print("\nCTI median từ Gold:", cti)

record(
    "CTI median",
    "PASS" if cti is not None and abs(cti - 3.26) < 0.005 else "NEEDS_REVIEW",
    actual=cti,
    expected=3.26,
    detail="Đối chiếu với KPI Catalog Chương 3; cần kiểm tra phạm vi nếu khác"
)

# 5. KIỂM TRA LỊCH SỬ THANH TOÁN
execute_test(
    "Installments source rows",
    "SELECT COUNT(*) FROM bronze.installments_payments",
    13605401
)

# 6. TỔNG HỢP THANH TOÁN ĐÚNG GRAIN KỲ
# Không cộng lặp AMT_INSTALMENT theo số lần thanh toán.
# Không thay thế AMT_PAYMENT thiếu bằng 0.

period_sql = """
WITH typed AS (
    SELECT
        TRY_CAST(SK_ID_PREV AS BIGINT) AS sk_id_prev,
        TRY_CAST(NUM_INSTALMENT_VERSION AS BIGINT) AS version,
        TRY_CAST(NUM_INSTALMENT_NUMBER AS BIGINT) AS installment_number,
        TRY_CAST(AMT_INSTALMENT AS DECIMAL(20,4)) AS due,
        TRY_CAST(AMT_PAYMENT AS DECIMAL(20,4)) AS paid,
        TRY_CAST(DAYS_INSTALMENT AS INTEGER) AS due_day,
        TRY_CAST(DAYS_ENTRY_PAYMENT AS INTEGER) AS paid_day
    FROM bronze.installments_payments
),
periods AS (
    SELECT
        sk_id_prev,
        version,
        installment_number,
        MAX(due) AS installment_due,
        SUM(paid) AS total_paid,
        COUNT(*) AS source_rows,
        COUNT(paid) AS valid_payment_rows,
        COUNT(DISTINCT due) AS distinct_due_count,
        COUNT(DISTINCT due_day) AS distinct_due_day_count,
        CASE
            WHEN COUNT(*) FILTER (
                WHERE paid_day > due_day
            ) > 0 THEN 1
            WHEN COUNT(*) = COUNT(paid_day)
             AND COUNT(*) = COUNT(due_day)
            THEN 0
            ELSE NULL
        END AS has_late_payment
    FROM typed
    GROUP BY 1,2,3
)
SELECT * FROM periods
"""

# Ghi ra Parquet trong vùng tạm Colab, không ghi vào DuckDB gốc.
period_file = Path("/content/dashboard_installment_periods.parquet")

con.execute(
    f"COPY ({period_sql}) TO '{period_file}' (FORMAT PARQUET)"
)

period_view = f"read_parquet('{period_file}')"

summary = con.execute(f"""
    SELECT
        COUNT(*) AS total_periods,

        COUNT(*) FILTER (
            WHERE valid_payment_rows = 0
        ) AS periods_without_valid_payment,

        COUNT(*) FILTER (
            WHERE distinct_due_count > 1
        ) AS inconsistent_due_periods,

        COUNT(*) FILTER (
            WHERE distinct_due_day_count > 1
        ) AS inconsistent_due_dates,

        COUNT(*) FILTER (
            WHERE has_late_payment = 1
        ) AS late_periods,

        COUNT(*) FILTER (
            WHERE has_late_payment IS NOT NULL
        ) AS valid_date_periods,

        COUNT(*) FILTER (
            WHERE installment_due > 0
              AND total_paid IS NOT NULL
              AND installment_due > total_paid
        ) AS positive_gap_periods,

        SUM(
            CASE
                WHEN installment_due > 0
                 AND total_paid IS NOT NULL
                 AND installment_due > total_paid
                THEN installment_due - total_paid
            END
        ) AS positive_payment_gap

    FROM {period_view}
""").df()

print("\n===== PAYMENT RECONCILIATION =====")
display(summary.T)

summary.to_csv(
    EVIDENCE / "payment_reconciliation.csv",
    index=False
)

row = summary.iloc[0]

checks = {
    "Installment period count": (
        "total_periods", 12951918, 0
    ),
    "Late installment periods": (
        "late_periods", 1103475, 0
    ),
    "Valid-date periods": (
        "valid_date_periods", 12949013, 0
    ),
    "Inconsistent due": (
        "inconsistent_due_periods", 0, 0
    ),
    "Inconsistent due dates": (
        "inconsistent_due_dates", 0, 0
    ),
    "Payment gap periods": (
        "positive_gap_periods", 2934, 0
    ),
    "Positive payment gap": (
        "positive_payment_gap", 21164943.06, 0.01
    )
}

for name, (column, expected, tolerance) in checks.items():
    actual = row[column]

    passed = (
        pd.notna(actual)
        and abs(float(actual) - expected) <= tolerance
    )

    record(
        name,
        "PASS" if passed else "NEEDS_REVIEW",
        actual=actual,
        expected=expected,
        detail="Đối chiếu baseline Chương 3"
    )

# 7. KIỂM TRA THỨ TỰ BẢN GHI XUẤT
# Chỉ kiểm tra file khi thực sự tồn tại trên Drive.
export_candidates = list(ROOT.rglob("ho_so_filtered.csv"))

if export_candidates:
    export_file = export_candidates[0]
    exported = pd.read_csv(export_file)

    record(
        "CSV export readable",
        "PASS",
        actual=len(exported),
        detail=str(export_file)
    )

    if "SK_ID_CURR" in exported.columns:
        duplicates = exported["SK_ID_CURR"].duplicated().sum()

        record(
            "CSV duplicate SK_ID_CURR",
            "PASS" if duplicates == 0 else "NEEDS_REVIEW",
            actual=int(duplicates),
            expected=0
        )

    else:
        record(
            "CSV key present",
            "FAIL",
            detail="Không có SK_ID_CURR"
        )

else:
    record(
        "CSV export",
        "NOT TESTED",
        detail="Chưa tìm thấy ho_so_filtered.csv trong Drive"
    )

# 8. XUẤT BẰNG CHỨNG
qa_df = pd.DataFrame(results)

qa_df.to_csv(
    EVIDENCE / "dashboard_qa_results.csv",
    index=False,
    encoding="utf-8-sig"
)

qa_metadata = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "database": str(DB),
    "database_mode": "read_only",
    "results": results
}

(EVIDENCE / "dashboard_qa_results.json").write_text(
    json.dumps(qa_metadata, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8"
)

con.close()

print("\n===== QA SUMMARY =====")
display(qa_df)

print("\nKết quả đã lưu tại:", EVIDENCE)

[PASS] Application Train: 307511
[PASS] Application Test: 48744
[PASS] Feature Mart: 307511
[PASS] Feature Mart duplicate SK_ID_CURR: 0
[PASS] TARGET = 1 rate (%): 8.072881945686495

CTI median từ Gold: 3.265066666666667
[NEEDS_REVIEW] CTI median: 3.265066666666667
[PASS] Installments source rows: 13605401


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


===== PAYMENT RECONCILIATION =====


,0
total_periods,12951918.00
periods_without_valid_payment,2905.00
inconsistent_due_periods,0.00
inconsistent_due_dates,0.00
late_periods,1103475.00
valid_date_periods,12949013.00
positive_gap_periods,2934.00
positive_payment_gap,21164943.06


[PASS] Installment period count: 12951918.0
[PASS] Late installment periods: 1103475.0
[PASS] Valid-date periods: 12949013.0
[PASS] Inconsistent due: 0.0
[PASS] Inconsistent due dates: 0.0
[PASS] Payment gap periods: 2934.0
[PASS] Positive payment gap: 21164943.06
[NOT TESTED] CSV export: None

===== QA SUMMARY =====


,test,status,actual,expected,detail
0,Application Train,PASS,3.075110e+05,3.075110e+05,
1,Application Test,PASS,4.874400e+04,4.874400e+04,
2,Feature Mart,PASS,3.075110e+05,3.075110e+05,
3,Feature Mart duplicate SK_ID_CURR,PASS,0.000000e+00,0.000000e+00,
4,TARGET = 1 rate (%),PASS,8.072882e+00,8.072900e+00,
5,CTI median,NEEDS_REVIEW,3.265067e+00,3.260000e+00,Đối chiếu với KPI Catalog Chương 3; cần kiểm t...
6,Installments source rows,PASS,1.360540e+07,1.360540e+07,
7,Installment period count,PASS,1.295192e+07,1.295192e+07,Đối chiếu baseline Chương 3
8,Late installment periods,PASS,1.103475e+06,1.103475e+06,Đối chiếu baseline Chương 3
9,Valid-date periods,PASS,1.294901e+07,1.294901e+07,Đối chiếu baseline Chương 3



Kết quả đã lưu tại: /content/drive/MyDrive/Data_Lakehouse_Project/evidence/dashboard


In [13]:
# ==============================================================================
# SCRIPT ĐỐI SOÁT ĐỘC LẬP SỐ LIỆU DASHBOARD 1 & DASHBOARD 2 (DÀNH CHO NGƯỜI 2)
# ==============================================================================
import os, sys, json
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

# 1. Kết nối Google Drive & Thiết lập đường dẫn Database
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/Data_Lakehouse_Project')
else:
    PROJECT_ROOT = Path.cwd()

# Tìm database DuckDB
db_candidates = [
    PROJECT_ROOT / 'database' / 'credit_lakehouse.duckdb',
    PROJECT_ROOT / 'data' / 'lakehouse_db' / 'credit_lakehouse.duckdb'
]
DB_PATH = next((p for p in db_candidates if p.exists()), None)
if DB_PATH is None:
    found = list(PROJECT_ROOT.rglob('credit_lakehouse.duckdb'))
    if found:
        DB_PATH = found[0]
    else:
        raise FileNotFoundError(f"Không tìm thấy file credit_lakehouse.duckdb tại {PROJECT_ROOT}")

print(f"✓ Đã kết nối DuckDB ở chế độ Read-Only: {DB_PATH}")
con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute("SET memory_limit='4GB'; SET threads=4;")

OUT_DIR = PROJECT_ROOT / 'evidence' / 'dashboard'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "="*80)
print("=== BẮT ĐẦU ĐỐI SOÁT SỐ LIỆU DASHBOARD 1 (APPLICATION RISK) ===")
print("="*80)

# ------------------------------------------------------------------------------
# DASHBOARD 1: KIỂM TRA 9 THÀNH PHẦN
# ------------------------------------------------------------------------------
# Ô 1 & 9: Tổng hồ sơ & Tỷ lệ có nhãn
df_d1_kpi1 = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM bronze.application_train) AS train_count,
        (SELECT COUNT(*) FROM bronze.application_test) AS test_count,
        (SELECT COUNT(*) FROM bronze.application_train) + (SELECT COUNT(*) FROM bronze.application_test) AS total_applications,
        ROUND(100.0 * (SELECT COUNT(*) FROM bronze.application_train) /
            ((SELECT COUNT(*) FROM bronze.application_train) + (SELECT COUNT(*) FROM bronze.application_test)), 4) AS labeled_rate_pct
""").df()

# Ô 2: Tỷ lệ TARGET = 1
df_d1_kpi2 = con.execute("""
    SELECT
        COUNT(*) AS total_train,
        COUNT(*) FILTER (WHERE TRY_CAST(TARGET AS INTEGER) = 1) AS target_1_count,
        COUNT(*) FILTER (WHERE TRY_CAST(TARGET AS INTEGER) = 0) AS target_0_count,
        ROUND(100.0 * COUNT(*) FILTER (WHERE TRY_CAST(TARGET AS INTEGER) = 1) / COUNT(*), 4) AS default_rate_pct
    FROM bronze.application_train
""").df()

# Ô 3, 4, 8: CTI, LTV Proxy (Credit/Goods), DTI trung vị & phân vị
df_d1_financials = con.execute("""
    WITH raw_calc AS (
        SELECT
            TRY_CAST(AMT_CREDIT AS DOUBLE) AS credit,
            TRY_CAST(AMT_INCOME_TOTAL AS DOUBLE) AS income,
            TRY_CAST(AMT_GOODS_PRICE AS DOUBLE) AS goods,
            TRY_CAST(AMT_ANNUITY AS DOUBLE) AS annuity
        FROM bronze.application_train
    )
    SELECT
        -- CTI
        ROUND(MEDIAN(credit / NULLIF(income, 0)), 4) AS cti_median,
        -- Credit / Goods (LTV Proxy)
        ROUND(MEDIAN(credit / NULLIF(goods, 0)), 4) AS credit_goods_median,
        COUNT(*) FILTER (WHERE goods IS NULL OR goods <= 0) AS missing_goods_count,
        -- DTI
        ROUND(MEDIAN(annuity / NULLIF(income, 0)) * 100, 2) AS dti_median_pct,
        ROUND(QUANTILE_CONT(annuity / NULLIF(income, 0), 0.25) * 100, 2) AS dti_q1_pct,
        ROUND(QUANTILE_CONT(annuity / NULLIF(income, 0), 0.75) * 100, 2) AS dti_q3_pct,
        COUNT(*) FILTER (WHERE annuity IS NULL) AS missing_annuity_count
    FROM raw_calc
""").df()

# Ô 5: TARGET = 1 theo nhóm Credit/Goods
df_d1_ltv_groups = con.execute("""
    WITH prep AS (
        SELECT
            TRY_CAST(TARGET AS INTEGER) AS target,
            TRY_CAST(AMT_CREDIT AS DOUBLE) / NULLIF(TRY_CAST(AMT_GOODS_PRICE AS DOUBLE), 0) AS ratio
        FROM bronze.application_train
        WHERE TRY_CAST(AMT_GOODS_PRICE AS DOUBLE) > 0
    )
    SELECT
        CASE
            WHEN ratio <= 0.5 THEN '1. <= 0.50'
            WHEN ratio <= 0.8 THEN '2. (0.50, 0.80]'
            WHEN ratio <= 1.0 THEN '3. (0.80, 1.00]'
            WHEN ratio <= 1.2 THEN '4. (1.00, 1.20]'
            WHEN ratio <= 1.5 THEN '5. (1.20, 1.50]'
            ELSE '6. > 1.50'
        END AS ltv_group,
        COUNT(*) AS total_records,
        SUM(target) AS target_1_count,
        ROUND(100.0 * SUM(target) / COUNT(*), 2) AS target_1_rate_pct
    FROM prep
    GROUP BY 1
    ORDER BY 1
""").df()

# Ô 6: TARGET = 1 theo nhóm tuổi
df_d1_age_groups = con.execute("""
    WITH age_calc AS (
        SELECT
            TRY_CAST(TARGET AS INTEGER) AS target,
            ABS(TRY_CAST(DAYS_BIRTH AS DOUBLE)) / 365.25 AS age_years
        FROM bronze.application_train
    )
    SELECT
        CASE
            WHEN age_years <= 30 THEN '1. 20-30 tuổi'
            WHEN age_years <= 45 THEN '2. 30-45 tuổi'
            WHEN age_years <= 60 THEN '3. 45-60 tuổi'
            ELSE '4. > 60 tuổi'
        END AS age_group,
        COUNT(*) AS total_records,
        SUM(target) AS target_1_count,
        ROUND(100.0 * SUM(target) / COUNT(*), 2) AS target_1_rate_pct
    FROM age_calc
    GROUP BY 1
    ORDER BY 1
""").df()

# Ô 7: Cơ cấu loại hợp đồng (Contract Type)
df_d1_contract = con.execute("""
    SELECT
        NAME_CONTRACT_TYPE,
        COUNT(*) AS total_records,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM bronze.application_train), 2) AS share_pct,
        SUM(TRY_CAST(TARGET AS INTEGER)) AS target_1_count,
        ROUND(100.0 * SUM(TRY_CAST(TARGET AS INTEGER)) / COUNT(*), 2) AS default_rate_pct
    FROM bronze.application_train
    GROUP BY NAME_CONTRACT_TYPE
""").df()

print("\n--- Ô 1, 2, 9: TỔNG QUAN HỒ SƠ & TARGET ---")
display(df_d1_kpi1)
display(df_d1_kpi2)

print("\n--- Ô 3, 4, 8: CTI, LTV PROXY & DTI TRUNG VỊ ---")
display(df_d1_financials)

print("\n--- Ô 5: TARGET = 1 THEO NHÓM CREDIT/GOODS ---")
display(df_d1_ltv_groups)

print("\n--- Ô 6: TARGET = 1 THEO NHÓM TUỔI ---")
display(df_d1_age_groups)

print("\n--- Ô 7: CƠ CẤU LOẠI HỢP ĐỒNG ---")
display(df_d1_contract)


print("\n" + "="*80)
print("=== BẮT ĐẦU ĐỐI SOÁT SỐ LIỆU DASHBOARD 2 (PAYMENT & BUREAU BEHAVIOR) ===")
print("="*80)

# ------------------------------------------------------------------------------
# DASHBOARD 2: KIỂM TRA 9 THÀNH PHẦN
# ------------------------------------------------------------------------------
# Ô 4: Tỷ lệ phê duyệt đơn cũ (previous_application)
df_d2_prev_status = con.execute("""
    SELECT
        NAME_CONTRACT_STATUS,
        COUNT(*) AS total_records,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM bronze.previous_application), 2) AS share_pct
    FROM bronze.previous_application
    GROUP BY NAME_CONTRACT_STATUS
    ORDER BY total_records DESC
""").df()

# Ô 5: Kỷ luật thanh toán theo ngày (cấp giao dịch nguồn installments_payments)
df_d2_tx_delay = con.execute("""
    WITH raw_dates AS (
        SELECT
            TRY_CAST(DAYS_ENTRY_PAYMENT AS DOUBLE) - TRY_CAST(DAYS_INSTALMENT AS DOUBLE) AS delay
        FROM bronze.installments_payments
        WHERE DAYS_ENTRY_PAYMENT IS NOT NULL AND DAYS_INSTALMENT IS NOT NULL
    )
    SELECT
        COUNT(*) AS valid_tx_count,
        COUNT(*) FILTER (WHERE delay < 0) AS early_tx_count,
        ROUND(100.0 * COUNT(*) FILTER (WHERE delay < 0) / COUNT(*), 2) AS early_pct,
        COUNT(*) FILTER (WHERE delay = 0) AS ontime_tx_count,
        ROUND(100.0 * COUNT(*) FILTER (WHERE delay = 0) / COUNT(*), 2) AS ontime_pct,
        COUNT(*) FILTER (WHERE delay > 0) AS late_tx_count,
        ROUND(100.0 * COUNT(*) FILTER (WHERE delay > 0) / COUNT(*), 2) AS late_pct,
        ROUND(MEDIAN(delay), 1) AS median_delay,
        ROUND(QUANTILE_CONT(delay, 0.95), 1) AS p95_delay,
        ROUND(QUANTILE_CONT(delay, 0.99), 1) AS p99_delay
    FROM raw_dates
""").df()

# Ô 1, 2, 3: Tổng hợp thanh toán theo Grain Kỳ & Cấp khách hàng
print("--> Đang tổng hợp 13.6M dòng thanh toán theo Grain Kỳ...")
df_d2_periods = con.execute("""
    WITH period_agg AS (
        SELECT
            SK_ID_CURR,
            SK_ID_PREV,
            NUM_INSTALMENT_VERSION,
            NUM_INSTALMENT_NUMBER,
            MAX(TRY_CAST(AMT_INSTALMENT AS DECIMAL(20,4))) AS installment_due,
            SUM(TRY_CAST(AMT_PAYMENT AS DECIMAL(20,4))) AS total_paid,
            MAX(TRY_CAST(DAYS_INSTALMENT AS DOUBLE)) AS due_day,
            MAX(TRY_CAST(DAYS_ENTRY_PAYMENT AS DOUBLE)) AS last_paid_day,
            COUNT(*) FILTER (WHERE TRY_CAST(DAYS_ENTRY_PAYMENT AS DOUBLE) > TRY_CAST(DAYS_INSTALMENT AS DOUBLE)) AS late_events
        FROM bronze.installments_payments
        GROUP BY SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER
    )
    SELECT
        COUNT(*) AS total_periods,
        -- Ô 1: Kỳ trễ hạn
        COUNT(*) FILTER (WHERE due_day IS NOT NULL AND last_paid_day IS NOT NULL) AS valid_date_periods,
        COUNT(*) FILTER (WHERE late_events > 0) AS late_periods,
        ROUND(100.0 * COUNT(*) FILTER (WHERE late_events > 0) /
            NULLIF(COUNT(*) FILTER (WHERE due_day IS NOT NULL AND last_paid_day IS NOT NULL), 0), 4) AS late_period_rate_pct,
        -- Ô 3: Thiếu hụt thanh toán
        COUNT(*) FILTER (WHERE installment_due > total_paid) AS shortfall_periods,
        ROUND(SUM(CASE WHEN installment_due > total_paid THEN installment_due - total_paid ELSE 0 END), 2) AS total_shortfall_amount
    FROM period_agg
""").df()

# Ô 2: Cấp khách hàng trả trễ (đối soát số lượng khách hàng chính xác)
df_d2_customer_late = con.execute("""
    WITH customer_late AS (
        SELECT
            SK_ID_CURR,
            MAX(CASE WHEN TRY_CAST(DAYS_ENTRY_PAYMENT AS DOUBLE) > TRY_CAST(DAYS_INSTALMENT AS DOUBLE) THEN 1 ELSE 0 END) AS ever_late
        FROM bronze.installments_payments
        WHERE DAYS_ENTRY_PAYMENT IS NOT NULL AND DAYS_INSTALMENT IS NOT NULL
        GROUP BY SK_ID_CURR
    )
    SELECT
        COUNT(*) AS total_customers_with_history,
        SUM(ever_late) AS customers_ever_late,
        ROUND(100.0 * SUM(ever_late) / COUNT(*), 4) AS customer_late_rate_pct
    FROM customer_late
""").df()

# Ô 7: TARGET = 1 theo mức dùng hạn mức thẻ tín dụng
df_d2_card_util = con.execute("""
    WITH card_customer AS (
        SELECT
            SK_ID_CURR,
            AVG(TRY_CAST(AMT_BALANCE AS DOUBLE) / NULLIF(TRY_CAST(AMT_CREDIT_LIMIT_ACTUAL AS DOUBLE), 0)) AS avg_utilization
        FROM bronze.credit_card_balance
        WHERE TRY_CAST(AMT_CREDIT_LIMIT_ACTUAL AS DOUBLE) > 0
        GROUP BY SK_ID_CURR
    ),
    card_with_target AS (
        SELECT
            c.avg_utilization,
            TRY_CAST(a.TARGET AS INTEGER) AS target
        FROM card_customer c
        INNER JOIN bronze.application_train a ON c.SK_ID_CURR = a.SK_ID_CURR
    )
    SELECT
        CASE
            WHEN avg_utilization <= 0 THEN '1. <= 0%'
            WHEN avg_utilization <= 0.3 THEN '2. (0-30%]'
            WHEN avg_utilization <= 0.5 THEN '3. (30-50%]'
            WHEN avg_utilization <= 0.8 THEN '4. (50-80%]'
            WHEN avg_utilization <= 1.0 THEN '5. (80-100%]'
            ELSE '6. > 100%'
        END AS util_group,
        COUNT(*) AS customer_count,
        SUM(target) AS target_1_count,
        ROUND(100.0 * SUM(target) / COUNT(*), 2) AS default_rate_pct
    FROM card_with_target
    GROUP BY 1
    ORDER BY 1
""").df()

# Ô 8: Cảnh báo sớm EWS (Thẻ, CIC quá hạn, Dư nợ ngoài)
df_d2_ews = con.execute("""
    SELECT
        (SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE TRY_CAST(AMT_BALANCE AS DOUBLE) / NULLIF(TRY_CAST(AMT_CREDIT_LIMIT_ACTUAL AS DOUBLE), 0) > 0.8) /
            NULLIF(COUNT(*), 0), 2)
         FROM bronze.credit_card_balance WHERE TRY_CAST(AMT_CREDIT_LIMIT_ACTUAL AS DOUBLE) > 0) AS card_over_80_pct,
        (SELECT COUNT(*) FILTER (WHERE TRY_CAST(CREDIT_DAY_OVERDUE AS INTEGER) > 0) FROM bronze.bureau) AS bureau_overdue_loans,
        (SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE TRY_CAST(CREDIT_DAY_OVERDUE AS INTEGER) > 0) / COUNT(*), 2) FROM bronze.bureau) AS bureau_overdue_rate_pct,
        (SELECT ROUND(SUM(TRY_CAST(AMT_CREDIT_SUM_DEBT AS DOUBLE)), 2) FROM bronze.bureau) AS total_bureau_debt
""").df()

print("\n--- Ô 1, 3: KỲ TRẢ TRỄ VÀ THIẾU HỤT THANH TOÁN ---")
display(df_d2_periods)

print("\n--- Ô 2: KHÁCH HÀNG TỪNG TRẢ TRỄ (CẤP KHÁCH HÀNG) ---")
display(df_d2_customer_late)

print("\n--- Ô 4: TRẠNG THÁI HỢP ĐỒNG CŨ (PREVIOUS APPLICATION) ---")
display(df_d2_prev_status)

print("\n--- Ô 5: KỶ LUẬT THANH TOÁN (CẤP GIAO DỊCH) ---")
display(df_d2_tx_delay)

print("\n--- Ô 7: MỨC DÙNG THẺ TÍN DỤNG VS TARGET = 1 ---")
display(df_d2_card_util)

print("\n--- Ô 8: CHỈ SỐ CẢNH BÁO SỚM (EWS) ---")
display(df_d2_ews)

# ------------------------------------------------------------------------------
# XUẤT TOÀN BỘ SỐ LIỆU ĐỐI SOÁT RA FILE EXCEL
# ------------------------------------------------------------------------------
excel_out = OUT_DIR / "Dashboard_Metrics_Reconciliation.xlsx"
with pd.ExcelWriter(excel_out, engine='openpyxl') as writer:
    df_d1_kpi1.to_excel(writer, sheet_name='D1_Volume_Target', index=False)
    df_d1_financials.to_excel(writer, sheet_name='D1_Financial_KPIs', index=False)
    df_d1_ltv_groups.to_excel(writer, sheet_name='D1_LTV_Risk', index=False)
    df_d1_age_groups.to_excel(writer, sheet_name='D1_Age_Risk', index=False)
    df_d1_contract.to_excel(writer, sheet_name='D1_Contract_Type', index=False)

    df_d2_periods.to_excel(writer, sheet_name='D2_Period_Shortfall', index=False)
    df_d2_customer_late.to_excel(writer, sheet_name='D2_Customer_Late', index=False)
    df_d2_prev_status.to_excel(writer, sheet_name='D2_Previous_Status', index=False)
    df_d2_tx_delay.to_excel(writer, sheet_name='D2_Tx_Discipline', index=False)
    df_d2_card_util.to_excel(writer, sheet_name='D2_Card_Util_Risk', index=False)
    df_d2_ews.to_excel(writer, sheet_name='D2_EWS_Metrics', index=False)

print("\n" + "="*80)
print(f"✓ ĐÃ HOÀN TẤT ĐỐI SOÁT! Toàn bộ bảng số liệu đã xuất tại: {excel_out}")
print("="*80)
con.close()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Đã kết nối DuckDB ở chế độ Read-Only: /content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb

=== BẮT ĐẦU ĐỐI SOÁT SỐ LIỆU DASHBOARD 1 (APPLICATION RISK) ===

--- Ô 1, 2, 9: TỔNG QUAN HỒ SƠ & TARGET ---


,train_count,test_count,total_applications,labeled_rate_pct
0,307511,48744,356255,86.3177


,total_train,target_1_count,target_0_count,default_rate_pct
0,307511,24825,282686,8.0729



--- Ô 3, 4, 8: CTI, LTV PROXY & DTI TRUNG VỊ ---


,cti_median,credit_goods_median,missing_goods_count,dti_median_pct,dti_q1_pct,dti_q3_pct,missing_annuity_count
0,3.2651,1.1188,278,16.28,11.48,22.91,12



--- Ô 5: TARGET = 1 THEO NHÓM CREDIT/GOODS ---


... (21 outputs left out)

,util_group,customer_count,target_1_count,default_rate_pct
0,1. <= 0%,26605,1438.0,5.40
1,2. (0-30%],19529,1257.0,6.44
2,3. (30-50%],12932,1068.0,8.26
3,4. (50-80%],16539,1885.0,11.40
4,5. (80-100%],9427,1527.0,16.20
5,6. > 100%,1004,256.0,25.50



--- Ô 8: CHỈ SỐ CẢNH BÁO SỚM (EWS) ---


,card_over_80_pct,bureau_overdue_loans,bureau_overdue_rate_pct,total_bureau_debt
0,28.77,4217,0.25,1.999742e+11



✓ ĐÃ HOÀN TẤT ĐỐI SOÁT! Toàn bộ bảng số liệu đã xuất tại: /content/drive/MyDrive/Data_Lakehouse_Project/evidence/dashboard/Dashboard_Metrics_Reconciliation.xlsx
